In [ ]:
# %pip uninstall -y numpy
# %pip install "numpy==2.2.4"


In [ ]:
# # !pip install scikit-image
# %pip install xgboost
# !pip list

In [ ]:
import numpy as np
import pandas as pd
import os
import glob
import re
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from TICC_solver import TICC
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import balanced_accuracy_score
# from skimage.filters import threshold_otsu

%matplotlib inline

In [ ]:
import random

np.random.seed(42)
random.seed(42)

In [ ]:
def _compute_window_band_powers(vals, fs, window_size, bands, center=True, pad=True, window_type='hann'):
    n = len(vals)
    if n == 0:
        return np.zeros((0, len(bands)))

    # 窓関数
    if window_type in ['hann', 'hanning']:
        win = np.hanning(window_size)
    else:
        win = np.ones(window_size)

    # 周波数軸（必ず window_size を使う）
    freqs = np.fft.rfftfreq(window_size, d=1.0/fs)

    band_mat = np.zeros((n, len(bands)), dtype=float)

    for i in range(n):
        # ウィンドウ位置決定
        if center:
            start = i - window_size // 2
        else:
            start = i
        end = start + window_size

        # 切り出し＋パディング
        if start < 0 or end > n:
            if pad:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                left = s_idx - start
                right = end - e_idx
                seg_padded = np.pad(seg, (max(0, left), max(0, right)), mode='reflect')

                if len(seg_padded) < window_size:
                    seg_padded = np.pad(seg_padded, (0, window_size - len(seg_padded)), mode='edge')

                windowed = seg_padded[:window_size]
            else:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                if len(seg) < window_size:
                    windowed = np.pad(seg, (0, window_size - len(seg)), mode='constant')
                else:
                    windowed = seg[:window_size]
        else:
            windowed = vals[start:end]

        # DC成分除去
        windowed = windowed - np.mean(windowed)

        # 窓関数適用
        windowed = windowed * win

        # --- FFT ---
        fft_vals = np.fft.rfft(windowed)

        # 振幅（正規化：2/N）
        amp = (2.0 / window_size) * np.abs(fft_vals)

        # パワー
        power = amp ** 2

        # --- バンドごとに合計 ---
        for b_i, (f_lo, f_hi) in enumerate(bands):
            mask = (freqs >= f_lo) & (freqs < f_hi)
            band_mat[i, b_i] = float(np.sum(power[mask])) if np.any(mask) else 0.0

    return band_mat

# 加速度前処理（特徴量作成）
def preprocess_acc_sensor_minimal(df_raw, sensor_suffix="acc",
                                  fs=100.0, window_size=1024,
                                  bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
                                  center=True, pad=True,
                                  window_stat=100):
    df = df_raw.copy()

    # ---- 時刻処理 ----
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce")
    df["ArriveTime"] = df["ArriveTime"].ffill()
    df["time"] = df["ArriveTime"]

    # ---- 加速度を float へ ----
    ax = pd.to_numeric(df.get("AccelerationX", 0), errors="coerce").fillna(0.0).astype(float)
    ay = pd.to_numeric(df.get("AccelerationY", 0), errors="coerce").fillna(0.0).astype(float)
    az = pd.to_numeric(df.get("AccelerationZ", 0), errors="coerce").fillna(0.0).astype(float)

    out = pd.DataFrame(index=df.index)

    # ---- 基本値 ----
    out[f"time_{sensor_suffix}"] = df["time"].values
    out[f"acc_x_{sensor_suffix}"] = ax
    out[f"acc_y_{sensor_suffix}"] = ay
    out[f"acc_z_{sensor_suffix}"] = az

    # ---- magnitude ----
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    out[f"acc_mag_{sensor_suffix}"] = mag

    # ---- 差分 ----
    out[f"acc_mag_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(mag)]
    out[f"acc_x_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(ax)]
    out[f"acc_y_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(ay)]
    out[f"acc_z_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(az)]

    # ============================================
    #   ★★★ 追加した特徴量 ★★★
    # ============================================

    # --- mean_x / mean_y / mean_z（移動平均） ---
    out[f"mean_x_{sensor_suffix}"] = ax.rolling(window_stat, min_periods=1).mean()
    out[f"mean_y_{sensor_suffix}"] = ay.rolling(window_stat, min_periods=1).mean()
    out[f"mean_z_{sensor_suffix}"] = az.rolling(window_stat, min_periods=1).mean()

    # --- std_x / std_y / std_z（移動 std） ---
    out[f"std_x_{sensor_suffix}"] = ax.rolling(window_stat, min_periods=1).std().fillna(0)
    out[f"std_y_{sensor_suffix}"] = ay.rolling(window_stat, min_periods=1).std().fillna(0)
    out[f"std_z_{sensor_suffix}"] = az.rolling(window_stat, min_periods=1).std().fillna(0)

    # --- mean_mag / std_mag ---
    out[f"mean_mag_{sensor_suffix}"] = mag.rolling(window_stat, min_periods=1).mean()
    out[f"std_mag_{sensor_suffix}"]  = mag.rolling(window_stat, min_periods=1).std().fillna(0)

    # --- SMA（瞬間値）---
    out[f"sma_{sensor_suffix}"] = (np.abs(ax) + np.abs(ay) + np.abs(az)) / 3.0

    # --- int_abs（mag の積分値）---
    # dt = 1.0 / fs
    # out[f"int_abs_{sensor_suffix}"] = np.cumsum(np.abs(mag)) * dt

    # ============================================
    #   ここまで追加特徴量
    # ============================================

    # ---- バンドパワー ----
    band_x = _compute_window_band_powers(ax.values, fs, window_size, bands, center, pad)
    band_y = _compute_window_band_powers(ay.values, fs, window_size, bands, center, pad)
    band_z = _compute_window_band_powers(az.values, fs, window_size, bands, center, pad)
    band_m = _compute_window_band_powers(mag, fs, window_size, bands, center, pad)

    for b_i, (f_lo, f_hi) in enumerate(bands):
        bi = b_i + 1
        out[f"acc_x_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_x[:, b_i]
        out[f"acc_y_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_y[:, b_i]
        out[f"acc_z_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_z[:, b_i]
        out[f"acc_mag_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_m[:, b_i]

    return out


def load_acc_only(
        keyword,
        root_folder=r"../../../../../デスクトップ/実験",
        fs=100.0,
        window_size=1024,
        bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
        center=True,
        pad=True,
        interpolate_missing=True,
        save_path=None):

    keyword = str(keyword)

    # 加速度フォルダ
    acc_path = os.path.join(root_folder, "data_acc")

    # ファイル検索：keyword + 10D739C を含むものだけ
    acc_all = glob.glob(os.path.join(acc_path, "*.csv"))
    acc_files = [
        f for f in acc_all
        if (keyword.lower() in os.path.basename(f).lower()) and
           ("10D739C".lower() in os.path.basename(f).lower())
    ]

    if len(acc_files) == 0:
        raise FileNotFoundError(f"keyword='{keyword}', '10D739C' を含む加速度ファイルが見つかりません")

    # 最初の 1 ファイルだけ使用
    acc_file = acc_files[0]
    print(f"Keyword: {keyword} -> Using: {os.path.basename(acc_file)}")

    # --- 加速度ロード ---
    df = pd.read_csv(acc_file)
    df.columns = df.columns.str.strip()

    # 時刻処理
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce").ffill()
    df["time"] = df["ArriveTime"]

    # 数値変換
    df["AccelerationX"] = pd.to_numeric(df.get("AccelerationX"), errors="coerce").fillna(0.0)
    df["AccelerationY"] = pd.to_numeric(df.get("AccelerationY"), errors="coerce").fillna(0.0)
    df["AccelerationZ"] = pd.to_numeric(df.get("AccelerationZ"), errors="coerce").fillna(0.0)

    # --- 特徴量計算 ---
    acc_feat = preprocess_acc_sensor_minimal(
        df, "acc", fs, window_size, bands, center, pad
    )

    # 欠損補完
    if interpolate_missing:
        acc_feat = acc_feat.interpolate("linear").ffill().bfill()

    # datetime 列を除去
    acc_feat = acc_feat.select_dtypes(include=["float", "int"]).ffill()

    # 最初の450データを落とす
    acc_feat = acc_feat.iloc[450:, :].reset_index(drop=True)

    # 標準化
    sc = StandardScaler()
    acc_feat = pd.DataFrame(sc.fit_transform(acc_feat), columns=acc_feat.columns)

    # 保存
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        acc_feat.to_csv(save_path, index=False)
        print("✔ 保存:", save_path)

    return acc_feat


def generate_log_bands_safe(fs=100, window=128, f_max=25.0, n_bands=12, base=10):
    delta_f = fs / window
    nyquist = fs / 2
    f_max = min(f_max, nyquist)

    # 生のログ境界
    raw_edges = np.logspace(np.log10(delta_f), np.log10(f_max), n_bands + 1, base=base)

    bands = []
    for i in range(n_bands):
        f_lo_raw, f_hi_raw = raw_edges[i], raw_edges[i+1]

        # FFT bin にスナップ
        f_lo_bin = int(np.floor(f_lo_raw / delta_f))
        f_hi_bin = int(np.ceil(f_hi_raw / delta_f))

        # 最低 1 bin を保証
        if f_hi_bin <= f_lo_bin:
            f_hi_bin = f_lo_bin + 1

        f_lo = f_lo_bin * delta_f
        f_hi = f_hi_bin * delta_f

        # Nyquist を越えたら調整
        if f_hi > nyquist:
            f_hi = nyquist
            f_lo = max(delta_f, f_hi - delta_f)

        bands.append((round(f_lo, 4), round(f_hi, 4)))

    return bands

def classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc):
    df.to_csv("data/no_heaer.csv", header=False, index=False, na_rep="NaN")
    
    # TICCインスタンスの作成
    ticc = TICC(window_size=window_size,
                number_of_clusters=number_of_clusters,
                beta=beta,
                maxIters=maxIters,
                threshold=2e-5,
                write_out_file=False,
                prefix_string="output_folder/",
                num_proc=num_proc)
    
    # クラスタリングを実行して cluster_assignment を取得
    cluster_assignment, cluster_MRFs = ticc.fit(input_file="data/no_heaer.csv")
    
    # cluster_assignmentの長さがdfと一致しない場合は先頭にNaNで埋める
    if len(cluster_assignment) != len(df):
        pad_len = len(df) - len(cluster_assignment)
        if pad_len > 0:
            # 先頭にnp.nanで埋める（または-1でもOK）
            cluster_assignment_padded = np.concatenate([np.full(pad_len, np.nan), cluster_assignment])
        else:
            raise ValueError(f"cluster_assignment length ({len(cluster_assignment)}) does not match df length ({len(df)})")
    else:
        cluster_assignment_padded = cluster_assignment

    df["class"] = cluster_assignment_padded

    df1 = df
    # df1.to_csv('data/data1_with_class.csv', index=False)
    
    return df1, cluster_assignment_padded

def plot_variables(df, variables,column = 'class',title=''):
    # color_map = {0: 'blue', 1: 'orange', 2: 'green', 3: 'red', 4: 'purple'}
    color_map = {0: 'blue', 1: 'orange', 4: 'green', 3: 'red', 2: 'purple', 5: 'brown'}

    cluster_series = df[column].values
    change_points = [0] + list(np.where(cluster_series[1:] != cluster_series[:-1])[0] + 1) + [len(df)]

    for var in variables:
        plt.figure(figsize=(20, 2))

        for start, end in zip(change_points[:-1], change_points[1:]):
            cluster = cluster_series[start]

            # NaN のクラスタはスキップ
            if pd.isna(cluster):
                continue

            color = color_map.get(cluster, 'gray')
            plt.plot(df.index[start:end], df[var].iloc[start:end], color=color)

        plt.xlabel('index', fontsize=12)
        # plt.ylabel(var, fontsize=12)
        plt.title(f'{var} with Cluster')
        plt.grid()
        plt.show()

def plot_variables_acc_label(df, variables,title=''):
    color_map = {0: 'blue', 1: 'orange', 4: 'green', 3: 'red', 2: 'purple', 5: 'brown'}
    cluster_series = df['acc_label'].values
    change_points = [0] + list(np.where(cluster_series[1:] != cluster_series[:-1])[0] + 1) + [len(df)]

    for var in variables:
        plt.figure(figsize=(20, 2))

        for start, end in zip(change_points[:-1], change_points[1:]):
            cluster = cluster_series[start]

            # NaN のクラスタはスキップ
            if pd.isna(cluster):
                continue

            color = color_map.get(cluster, 'gray')
            plt.plot(df.index[start:end], df[var].iloc[start:end], color=color)

        plt.xlabel('index')
        plt.ylabel(var)
        plt.title(f'{var} with Label {title}')
        plt.grid()
        plt.show()

def clustering_accuracy_per_class(df, true_col='acc_label', cluster_col='class'):
    true_labels = df[true_col].to_numpy()
    cluster_labels = df[cluster_col].to_numpy()

    clusters = np.unique(cluster_labels)
    classes = np.unique(true_labels)

    # --- Hungarian のためのコスト行列 ---
    cost_matrix = np.zeros((len(clusters), len(classes)))
    for i, c in enumerate(clusters):
        for j, t in enumerate(classes):
            cost_matrix[i, j] = -np.sum((cluster_labels == c) & (true_labels == t))

    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    # --- クラスタ → 真のラベルの対応 ---
    mapping = {clusters[r]: classes[c] for r, c in zip(row_idx, col_idx)}

    # --- 予測ラベル ---
    predicted = np.array([mapping[c] for c in cluster_labels])

    # --- 全体精度 ---
    overall_acc = accuracy_score(true_labels, predicted)

    # --- クラス別精度 ---
    per_class_acc = {}
    for cls in classes:
        idx = np.where(true_labels == cls)[0]
        per_class_acc[cls] = accuracy_score(true_labels[idx], predicted[idx])

    return overall_acc, per_class_acc, mapping, predicted

In [ ]:
keywords = ['kannno1', 'sato1', 'nishi2','mori1','mori2',"tanaka1","tanaka2","tuji1","tuji2","okawa1","okawa2","maeda1","maeda2"]
# keywords = ['kannno1']

windows = [512]

# 結果格納用
dfs = {}

for wd in windows:
    print(f"------- window_size={wd} ------------")
    bands = generate_log_bands_safe(
        window=wd,
        f_max=25.0,
        n_bands=10
    )

    for kw in keywords:
        # 1ファイルだけ処理
        df_key = f"df{kw}_{wd}"
        dfs[df_key] = load_acc_only(
            keyword=kw,
            window_size=wd,
            bands=bands
        )


In [ ]:
num_proc = 2     # 並列化（CPUが12コア）
window_size = 1
number_of_clusters = 2
beta = 25
maxIters = 30

# keyword = keywords[8]
keyword = keywords[0]

for wd in windows:
    for kw in keywords:
        print(f"------- window_size={wd} ------------")
        print(f"--- keyword={kw} ---")
        # クラスタリング
        df = dfs[f"df{kw}_{wd}"]
        classified_df,df_class = classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc)
        df["class"] = df_class

        dfs[f"df{kw}_{wd}"] = df

In [ ]:
# 　精度評価
f1_scores = {}

for wd in windows:
    for kw in keywords:
        print(f"------- window_size={wd} ------------")
        print(f"--- keyword={kw} ---")
        # クラスタリング結果取得
        df = dfs[f"df{kw}_{wd}"]

        df_class = df["class"]
        df_label = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")
        df = pd.concat([df_class, df_label], axis=1)
	
        df = df.dropna(subset=['acc_label','class'])
    
        # --- 精度計算 ---
        accuracy, per_class_acc, mapping, predicted_labels = clustering_accuracy_per_class(df)

        true_labels = df["acc_label"].to_numpy()

        # --- クラス出現数 ---
        unique, counts = np.unique(true_labels, return_counts=True)

        # 多数派 & 少数派クラス
        class_counts = dict(zip(unique, counts))
        majority_class = max(class_counts, key=class_counts.get)
        minority_class = min(class_counts, key=class_counts.get)

        print(f"Overall Accuracy: {accuracy:.4f}")
        print(f"Per Class Accuracy: {per_class_acc}")
        print(f"Class Counts: {class_counts} (Majority: {majority_class}, Minority: {minority_class})")
        print("f1_score:", f1_score(true_labels, predicted_labels, average='weighted'))
        
        f1_scores[f"{kw}_{wd}"] = f1_score(true_labels, predicted_labels, average='weighted')
        

        # plot_variables(df, variables=["acc_x_acc"], column='class', title=f"{kw} TICC")

        # 多数派クラスのclassが1になるように調整
        dfs[f"df{kw}_{wd}"]['class'] = dfs[f"df{kw}_{wd}"]['class'].map(mapping)
        plot_variables(dfs[f"df{kw}_{wd}"], variables=["acc_x_acc"], column='class', title=f"{kw} TICC")


In [ ]:
def classify_random_forest(dfs,keyword, window,class_col = "class"):
    # 特徴量とラベルの抽出
    df = dfs[f"df{keyword}_{window}"]

    # NaN値を除去してから学習用データを準備
    df_train = df.dropna(subset=[class_col])
    y = df_train[class_col].values
    X = df_train.drop(columns=[class_col]).values

    # ランダムフォレスト分類器の作成
    clf = RandomForestClassifier(n_estimators=100, random_state=42,max_depth=3)

    # モデルの学習
    clf.fit(X, y)

    return clf, clf.feature_importances_

In [ ]:

keywords = ['kannno1', 'sato1', 'nishi2','mori1','mori2',"tanaka1","tanaka2","tuji1","tuji2","okawa1","okawa2","maeda1","maeda2"]
keyword = keywords[0]
clf, feature_importances = classify_random_forest(dfs, keyword,window=512)

# x = ['acc_x_band4_0.7812-1.3672Hz', 'acc_x_band3_0.3906-0.9766Hz', 'acc_x_band1_0.1953-0.3906Hz', 'acc_x_band2_0.1953-0.5859Hz', 'acc_z_band3_0.3906-0.9766Hz', 'mean_x', 'acc_y_band5_1.1719-2.3438Hz', 'acc_y_band1_0.1953-0.3906Hz', 'acc_y_band8_5.6641-9.5703Hz', 'acc_y_band2_0.1953-0.5859Hz', 'std_y', 'acc_z_band4_0.7812-1.3672Hz', 'mean_y', 'acc_x', 'acc_y_band4_0.7812-1.3672Hz', 'acc_y_band6_2.1484-3.7109Hz', 'acc_z_band1_0.1953-0.3906Hz', 'acc_z_band2_0.1953-0.5859Hz', 'acc_y_band10_15.2344-25.1953Hz', 'acc_y_band9_9.375-15.4297Hz']
# 特徴量重要度の上位をグラフ表示
importances = pd.Series(feature_importances, index=dfs[f"df{keyword}_{windows[0]}"].drop(columns=["class"]).columns)

# 上位10個をimpに格納
imp = importances.sort_values(ascending=False).head(10)
# impのインデックスをxに置き換え
# imp.index = x
imp.plot(kind='barh', figsize=(8,6))
plt.title(f"Top 10 Feature Importances",fontsize=15)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

print(imp.index.tolist())

In [ ]:
# 重要変数上位1つの時系列をプロット
top1_var = importances.sort_values(ascending=False).head(1).index.tolist()
plot_variables(dfs[f"df{keyword}_{windows[0]}"], variables=top1_var, column='class', title=f"{keyword}")


# １本にしてTICC

# keywordsの時系列を１本にまとめてクラスタリング
# keywords = ['kannno1','sato1','nishi2', 'mori1','mori2']

combined_dfs = []
for kw in keywords:
    df = dfs[f"df{kw}_{windows[0]}"]
    df = df.drop(columns=["class"])
    combined_dfs.append(df)

combined_df = pd.concat(combined_dfs, ignore_index=True)

print(f"--- Combined Clustering for All Keywords ---")

# クラスタリング
classified_df,df_class = classify_time_series(combined_df, window_size, number_of_clusters, beta, maxIters, num_proc)
combined_df["class"] = df_class
dfs["combined_df"] = combined_df

# 　精度評価
df = dfs["combined_df"]
df_class = df["class"]

df_label_list = []
for kw in keywords:
    df_label = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")
    df_label_list.append(df_label)
df_label = pd.concat(df_label_list, ignore_index=True)

df = pd.concat([df_class, df_label], axis=1)
# acc_label列またはclass列にNaNがある行を削除（NaNクラスタがあると KeyError: np.float64(nan) になるため）
df = df.dropna(subset=['acc_label', 'class'])
# --- 精度計算 ---
accuracy, per_class_acc, mapping, predicted_labels = clustering_accuracy_per_class(df)
true_labels = df["acc_label"].to_numpy()
# --- クラス出現数 ---
unique, counts = np.unique(true_labels, return_counts=True)
# 多数派 & 少数派クラス
class_counts = dict(zip(unique, counts))
majority_class = max(class_counts, key=class_counts.get)
minority_class = min(class_counts, key=class_counts.get)
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"Per Class Accuracy: {per_class_acc}")
print(f"Class Counts: {class_counts} (Majority: {majority_class}, Minority: {minority_class})")

plot_variables(df, variables=["acc_x_acc1"], column='class', title=f"All Keywords Combined TICC")
plot_variables_acc_label(df, variables=["acc_x_acc1"], title=f"All Keywords Combined True Labels")



# combined_dfを保存

save_root = "../edgeAI実装/class_data_acc"   # ← 後で好きなパスに変更
import os

os.makedirs(save_root, exist_ok=True)
save_path = os.path.join(save_root, "combined_df_with_class.csv")
dfs["combined_df"].to_csv(save_path, index=False)
print("✔ 保存:", save_path)


# 複数モデル

In [ ]:
def run_model_safe(df_train, train_labels, df_test, test_labels, model, verbose=True):
    train_labels = np.asarray(train_labels)
    test_labels = np.asarray(test_labels)

    # --- Train NaN除外 ---
    mask_train = ~np.isnan(train_labels)
    df_train_clean = df_train.iloc[mask_train.nonzero()[0]]
    train_labels_clean = train_labels[mask_train]

    if len(df_train_clean) == 0:
        raise ValueError("Train data is empty after removing NaN labels")

    # --- Drop class column ---
    # if 'class' in df_train_clean.columns:
    #     df_train_clean = df_train_clean.drop(columns=['class'])
    # df_test_clean = df_test.drop(columns=['class'], errors='ignore')
    
    # 学習データとテストデータから、classという文字列を含むカラムをすべて削除
    cols_to_drop = [c for c in df_train_clean.columns if "class" in c]
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean = df_test.drop(columns=cols_to_drop, errors='ignore')
    
    # --- Fit model ---
    model.fit(df_train_clean, train_labels_clean)

    # --- Predict ---
    y_pred_full = model.predict(df_test_clean)

    # --- Length mismatch warning ---
    if len(y_pred_full) != len(test_labels):
        if verbose:
            print(f"Warning: Length mismatch -> y_pred: {len(y_pred_full)}, test_labels: {len(test_labels)}")

    min_len = min(len(y_pred_full), len(test_labels))
    acc = accuracy_score(test_labels[:min_len], y_pred_full[:min_len])

    return acc, y_pred_full, model

def build_model(model_name):
    model_name = model_name.lower()

    if model_name == "rf":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced",
            random_state=42
        )

    elif model_name == "xgb":
        from xgboost import XGBClassifier
        return XGBClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=10,
            random_state=42,
            eval_metric="logloss"
        )

    elif model_name == "cat":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            iterations=300,
            learning_rate=0.1,
            depth=10,
            random_seed=42,
            verbose=False
        )

    else:
        raise ValueError(f"Unknown model name: {model_name}")


def run_model_safe_mode(df_train, train_labels, df_test, test_labels, model_name, verbose=True):
    model = build_model(model_name)
    return run_model_safe(df_train, train_labels, df_test, test_labels, model, verbose)


def evaluate_all_keywords(train_kw, dfs, predicted_labels, window, keywords, model_name):
    model = build_model(model_name)

    per_person_results = {}
    all_y_true = []
    all_y_pred = []

    df_train = dfs[f"df{train_kw}_{window}"]
    y_train = np.asarray(predicted_labels[f"df{train_kw}_{window}"])

    for test_kw in keywords:
        if test_kw == train_kw:
            continue

        df_test = dfs[f"df{test_kw}_{window}"]
        y_test = np.asarray(predicted_labels[f"df{test_kw}_{window}"])

        min_len = min(len(df_test), len(y_test))
        df_test = df_test.iloc[-min_len:, :].reset_index(drop=True)
        y_test = y_test[-min_len:]

        # ←ここがモデル名で切り替わる
        acc, y_pred, _ = run_model_safe(df_train, y_train, df_test, y_test, model=model)

        per_person_results[test_kw] = {
            "accuracy": acc,
            "f1_macro": f1_score(y_test, y_pred, average="macro"),
            "confusion_matrix": confusion_matrix(y_test, y_pred),
            "y_true": y_test,
            "y_pred": y_pred
        }

        all_y_true.append(y_test)
        all_y_pred.append(y_pred)

    # overall 集計
    if all_y_true:
        all_y_true = np.concatenate(all_y_true)
        all_y_pred = np.concatenate(all_y_pred)

        overall = {
            "accuracy": accuracy_score(all_y_true, all_y_pred),
            "f1_macro": f1_score(all_y_true, all_y_pred, average="macro"),
            "confusion_matrix": confusion_matrix(all_y_true, all_y_pred)
        }
    else:
        overall = None

    return per_person_results, overall


# ランダムフォレスト

In [ ]:
# --- predicted_labels を安全に作成 ---
predicted_labels = {}
for kw in keywords:
    df_key = f"df{kw}_{windows[0]}"
    df = dfs[df_key]

    try:
        df_label = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")
        labels = df_label["acc_label"].to_numpy()  # ←必ず NumPy 配列
    except Exception:
        labels = np.full(len(df), np.nan)

    min_len = min(len(df), len(labels))
    predicted_labels[df_key] = np.asarray(labels[:min_len])  # ←念のため np.asarray()


# 複数モデル比較

In [ ]:
model_list = ["rf", "xgb"]

# results_all_models = {}

# for model_name in model_list:
#     print(f"\n==============================")
#     print(f" Evaluating model: {model_name}")
#     print(f"==============================")

#     for train_kw in keywords:
#         print(f"\n--- Training on {train_kw} ---")
        
#         per_person_results, overall_results = evaluate_all_keywords(
#             train_kw, dfs, predicted_labels, windows[0], keywords,
#             model_name=model_name
#         )

#         results_all_models[model_name] = {
#             "per_person": per_person_results,
#             "overall": overall_results
#         }

#         # 表示
#         for kw, res in per_person_results.items():
#             print(f"\nTest {kw} ({model_name})")
#             print(f"  Acc: {res['accuracy']:.4f}")
#             print(f"  F1 : {res['f1_macro']:.4f}")

#         print("\n--- Overall ---")
#         print(f"Accuracy: {overall_results['accuracy']:.4f}")
#         print(f"F1 (macro): {overall_results['f1_macro']:.4f}")
#         print(overall_results["confusion_matrix"])
        
#         # dfsに予測ラベルを追加
#         for test_kw in keywords:
#             if test_kw == train_kw:
#                 continue

#             df_key = f"df{test_kw}_{windows[0]}"
#             y_pred = per_person_results[test_kw]["y_pred"]


# 修正例: results_all_models を蓄積する形に変更
results_all_models = {}  # 初期化（上書きを避けるため、ループ外で1回だけ）

for model_name in model_list:
    for train_kw in keywords:
        print(f"\n--- Training on {train_kw} with {model_name} ---")
        
        per_person_results, overall_results = evaluate_all_keywords(
            train_kw, dfs, predicted_labels, windows[0], keywords, model_name=model_name
        )
        
        # 結果を蓄積（上書きせず追加）
        if model_name not in results_all_models:
            results_all_models[model_name] = {}
        results_all_models[model_name][train_kw] = {
            "per_person": per_person_results,
            "overall": overall_results
        }
        
        # 表示（必要に応じて）
        for kw, res in per_person_results.items():
            print(f"Test {kw}: Acc={res['accuracy']:.4f}, F1={res['f1_macro']:.4f}")
        print(f"Overall: Acc={overall_results['accuracy']:.4f}, F1={overall_results['f1_macro']:.4f}")
        
        # dfs に予測ラベルを追加（必要に応じて）
        for test_kw in keywords:
            if test_kw == train_kw:
                dfs[f"df{test_kw}_{windows[0]}"][f"{model_name}_class"] = dfs[f"df{test_kw}_{windows[0]}"]["class"]
            else:
                y_pred = per_person_results[test_kw]["y_pred"]
                dfs[f"df{test_kw}_{windows[0]}"][f"{model_name}_class"] = y_pred


In [ ]:
# F1スコアの表示
print("\n=== F1 Scores for Each Model and Training Keyword ===")
for model_name, model_results in results_all_models.items():
    print(f"\n--- Model: {model_name} ---")
    for train_kw, res in model_results.items():
        overall_f1 = res["overall"]["f1_macro"]
        print(f"Train on {train_kw}: Overall F1 (macro) = {overall_f1:.4f}")
    # 全体平均表示
    overall_f1s = [res["overall"]["f1_macro"] for res in model_results.values() if res["overall"] is not None]
    if overall_f1s:
        avg_f1 = np.mean(overall_f1s)
        print(f"Average Overall F1 (macro) across all training keywords: {avg_f1:.4f}")
        

In [ ]:
# モデルごと、訓練者ごとの macro F1（Overall）を、RFとXGBで並べた比較図プロット

# プロット用データの準備とプロット
model_f1_scores = {model: [] for model in model_list}
for model_name in model_list:
    for train_kw in keywords:
        overall = results_all_models[model_name][train_kw]["overall"]
        model_f1_scores[model_name].append(overall["f1_macro"])

# プロット（#VSC-d2f7e318 のコードを流用）
x = np.arange(len(keywords))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, model_f1_scores["rf"], width, label='Random Forest')
rects2 = ax.bar(x + width/2, model_f1_scores["xgb"], width, label='XGBoost')
ax.set_ylabel('Macro F1 Score')
ax.set_title('Macro F1 Score by Training Keyword and Model')
# x軸のラベルをkeywordsのインデックスに変更


ax.set_xticks(x)
ax.set_xticklabels([f"{i+1}" for i in range(len(keywords))], rotation=45)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# 訓練者のクラスタリング精度 vs RFの割り当てF1スコアの散布図プロット
rf_f1_scores = []
clustering_accuracies = []
for kw in keywords:
    # クラスタリング精度
    clustering_acc = f1_scores[f"{kw}_{windows[0]}"]
    clustering_accuracies.append(clustering_acc)

    # RFの割り当てF1スコア
    overall = results_all_models["rf"][kw]["overall"]
    rf_f1 = overall["f1_macro"]
    rf_f1_scores.append(rf_f1)
    
# 散布図プロット
plt.figure(figsize=(8, 6))
plt.scatter(clustering_accuracies, rf_f1_scores)
plt.xlabel('Clustering F1 Score',fontsize=12)
plt.ylabel('Random Forest Assignment F1 Score',fontsize=12)
plt.title('Clustering F1 Score vs Random Forest Assignment F1 Score',fontsize=15)
# for i, kw in enumerate(keywords):
#     plt.annotate(kw, (clustering_accuracies[i], rf_f1_scores[i]))
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid()
plt.show()

print("平均クラスタリングF1スコア:", np.mean(clustering_accuracies))
print("平均RF割り当てF1スコア:", np.mean(rf_f1_scores))

In [ ]:
# 訓練者のクラスタリング精度 vs XGBの割り当てF1スコアの散布図プロット
xgb_f1_scores = []
clustering_accuracies = []
for kw in keywords:
    # クラスタリング精度
    clustering_acc = f1_scores[f"{kw}_{windows[0]}"]
    clustering_accuracies.append(clustering_acc)

    # XGBの割り当てF1スコア
    overall = results_all_models["xgb"][kw]["overall"]
    xgb_f1 = overall["f1_macro"]
    xgb_f1_scores.append(xgb_f1)

# 散布図プロット
plt.figure(figsize=(8, 6))
plt.scatter(clustering_accuracies, xgb_f1_scores)
plt.xlabel('Clustering F1 Score',fontsize=12)
plt.ylabel('XGBoost Assignment F1 Score',fontsize=12)
plt.title('Clustering F1 Score vs XGBoost Assignment F1 Score',fontsize=15)
# for i, kw in enumerate(keywords):
#     plt.annotate(kw, (clustering_accuracies[i], xgb_f1_scores[i]))
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid()
plt.show()


# 保存用データ

model_name = 'rf'
results_all_models = {}

train_kw = 'tuji1'
# train_kw = 'tuji2'
# train_kw = 'kannno1'

print(f"\n--- Training on {train_kw} ---")
        
per_person_results, overall_results = evaluate_all_keywords(
            train_kw, dfs, predicted_labels, windows[0], keywords,
            model_name=model_name
)

results_all_models[model_name] = {
        "per_person": per_person_results,
        "overall": overall_results
    }

    # 表示
for kw, res in per_person_results.items():
    print(f"\nTest {kw} ({model_name})")
    print(f"  Acc: {res['accuracy']:.4f}")
    print(f"  F1 : {res['f1_macro']:.4f}")

print("\n--- Overall ---")
print(f"Accuracy: {overall_results['accuracy']:.4f}")
print(f"F1 (macro): {overall_results['f1_macro']:.4f}")
print(overall_results["confusion_matrix"])
        
# dfsに予測ラベルを追加
for test_kw in keywords:
    df_key = f"df{test_kw}_{windows[0]}"

    if test_kw == train_kw:
        # 教師本人：真のクラスをそのまま入れる
        dfs[df_key][f"{model_name}_class"] = dfs[df_key]["class"]
    else:
        # テスト対象：予測結果を入れる
        y_pred = per_person_results[test_kw]["y_pred"]
        dfs[df_key][f"{model_name}_class"] = y_pred


save_root = "../edgeAI実装/class_data_acc/data_last"   # ← 後で好きなパスに変更
import os

os.makedirs(save_root, exist_ok=True)
for kw in keywords:
    save_path = os.path.join(save_root, f"data_acc_{kw}_512.csv")
    dfs[f"df{kw}_512"].to_csv(save_path, index=False)
    print(f"✔ 保存: {save_path}")


keywords = ['tanaka1','tanaka2','tuji1','tuji2']

# 4人それぞれでrandomフォレストの分類器を学習し、他の3人を予測し、その結果を新しいdf_resultsに訓練者、予測クラス、正解ラベルとしてまとめた後、それを表示する

df_results = pd.DataFrame(columns=['train_keyword', 'test_keyword', 'predicted_class', 'true_label'])
for train_kw in keywords:
    print(f"\n--- Training on {train_kw} ---")
    
    per_person_results, overall_results = evaluate_all_keywords(
                train_kw, dfs, predicted_labels, windows[0], keywords,
                model_name='rf'
    )
    
    for test_kw in keywords:
        if test_kw == train_kw:
            continue
        
        res = per_person_results[test_kw]
        y_pred = res["y_pred"]
        y_true = res["y_true"]
        
        df_temp = pd.DataFrame({
            'train_keyword': [train_kw] * len(y_pred),
            'test_keyword': [test_kw] * len(y_pred),
            'predicted_class': y_pred,
            'true_label': y_true
        })
        
        df_results = pd.concat([df_results, df_temp], ignore_index=True)
print(df_results)


# 各訓練者ごとの予測精度を表示
for train_kw in keywords:
    df_train_results = df_results[df_results['train_keyword'] == train_kw]
    acc = accuracy_score(df_train_results['true_label'], df_train_results['predicted_class'])
    f1 = f1_score(df_train_results['true_label'], df_train_results['predicted_class'], average='macro')
    print(f"\nResults when trained on {train_kw}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")

# 全体の予測精度を表示
overall_acc = accuracy_score(df_results['true_label'], df_results['predicted_class'])
overall_f1 = f1_score(df_results['true_label'], df_results['predicted_class'], average='macro')
print(f"\nOverall Results:")
print(f"  Accuracy: {overall_acc:.4f}")
print(f"  F1 Score: {overall_f1:.4f}")


In [ ]:
dfs[f"df{keywords[0]}_{windows[0]}"].columns

In [ ]:
# k-means,GMMクラスタリングして、TICCのクラスタリング精度とk-means,GMM精度を比較する
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

kmeans_results = {}
gmm_results = {}

for kw in keywords:
    df = dfs[f"df{kw}_{windows[0]}"]
    X = df.drop(columns=['class', 'rf_class','xgb_class']).values
    kmeans = KMeans(n_clusters=number_of_clusters, random_state=42)
    kmeans_labels = kmeans.fit_predict(X)
    kmeans_results[kw] = kmeans_labels

    # GMMクラスタリング
    gmm = GaussianMixture(n_components=number_of_clusters, random_state=42)
    gmm_labels = gmm.fit_predict(X)
    gmm_results[kw] = gmm_labels
    
    # Hungarianアルゴリズムでクラスタと真のラベルの最適なマッピングを見つける
    true_labels = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")["acc_label"].to_numpy()
    cost_matrix_kmeans = np.zeros((number_of_clusters, len(np.unique(true_labels))))
    cost_matrix_gmm = np.zeros((number_of_clusters, len(np.unique(true_labels))))
    for i in range(number_of_clusters):
        for j in range(len(np.unique(true_labels))):
            cost_matrix_kmeans[i, j] = -np.sum((kmeans_labels == i) & (true_labels == j))
            cost_matrix_gmm[i, j] = -np.sum((gmm_labels == i) & (true_labels == j))

    row_idx_kmeans, col_idx_kmeans = linear_sum_assignment(cost_matrix_kmeans)
    row_idx_gmm, col_idx_gmm = linear_sum_assignment(cost_matrix_gmm)

    # マッピングに基づいてラベルを再割り当て
    kmeans_labels_mapped = np.zeros_like(kmeans_labels)
    gmm_labels_mapped = np.zeros_like(gmm_labels)
    for r, c in zip(row_idx_kmeans, col_idx_kmeans):
        kmeans_labels_mapped[kmeans_labels == r] = c
    for r, c in zip(row_idx_gmm, col_idx_gmm):
        gmm_labels_mapped[gmm_labels == r] = c
    

    # # TICCのクラスタリング結果とf1score精度比較
    # ticc_labels = df["class"].values
    # true_labels = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")["acc_label"].to_numpy()
    # ticc_f1 = f1_score(true_labels, ticc_labels, average='weighted')
    # kmeans_f1 = f1_score(true_labels, kmeans_labels, average='weighted')
    # gmm_f1 = f1_score(true_labels, gmm_labels, average='weighted')
    
    # TICCのクラスタリング結果とf1score精度比較（マッピング後）
    ticc_labels = df["class"].values
    ticc_f1 = f1_score(true_labels, ticc_labels, average='weighted')
    kmeans_f1 = f1_score(true_labels, kmeans_labels_mapped, average='weighted')
    gmm_f1 = f1_score(true_labels, gmm_labels_mapped, average='weighted')

    # # BA (Balanced Accuracy)の計算
    # ticc_ba = balanced_accuracy_score(true_labels, ticc_labels)
    # kmeans_ba = balanced_accuracy_score(true_labels, kmeans_labels)
    # gmm_ba = balanced_accuracy_score(true_labels, gmm_labels)

    # BA (Balanced Accuracy)の計算（マッピング後）
    ticc_ba = balanced_accuracy_score(true_labels, ticc_labels)
    kmeans_ba = balanced_accuracy_score(true_labels, kmeans_labels_mapped)
    gmm_ba = balanced_accuracy_score(true_labels, gmm_labels_mapped)

    # 結果表示
    print(f"{kw} - TICC F1: {ticc_f1:.4f}, KMeans F1: {kmeans_f1:.4f}, GMM F1: {gmm_f1:.4f}")
    print(f"{kw} - TICC BA: {ticc_ba:.4f}, KMeans BA: {kmeans_ba:.4f}, GMM BA: {gmm_ba:.4f}")


In [ ]:
# データ長の確認
for kw in keywords:
    df = dfs[f"df{kw}_{windows[0]}"]
    print(f"{kw}: Data length: {len(df)}")

In [ ]:
# true_labelsのクラスごとの長さ確認
for kw in keywords:
    true_labels = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")["acc_label"].to_numpy()
    unique_classes = np.unique(true_labels)
    print(f"{kw}:")
    for cls in unique_classes:
        count = np.sum(true_labels == cls)
        print(f"  Class {cls}: {count}")

In [ ]:
# TICC,k-means,GMMの結果の表示
for kw in keywords:
    df = dfs[f"df{kw}_{windows[0]}"]
    true_labels = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")["acc_label"].to_numpy()
    ticc_labels = df["class"].values
    kmeans_labels = kmeans_results[kw]
    gmm_labels = gmm_results[kw]

    # クラスごとの分布確認
    print(f"{kw} - True Labels Distribution:")
    print(np.bincount(true_labels.astype(int)))
    print(f"{kw} - TICC Cluster Distribution:")
    print(np.bincount(ticc_labels.astype(int)))
    print(f"{kw} - KMeans Cluster Distribution:")
    print(np.bincount(kmeans_labels.astype(int)))
    print(f"{kw} - GMM Cluster Distribution:")
    print(np.bincount(gmm_labels.astype(int)))